# ISRIC SoilGrids — Global 250 m Soil Properties (fallback)

**What it is.** Machine-learning global soil property maps at 250 m. For the U.S., **SSURGO
is more authoritative** (see `soil_ssurgo.ipynb`) — SoilGrids is the **global fallback** and a
cross-check.

| | |
|---|---|
| **Coverage** | Global, 250 m |
| **Depths** | 0-5, 5-15, 15-30, 30-60, 60-100, 100-200 cm |
| **Cost / key** | **Free · no key** |
| **Docs** | https://rest.isric.org/soilgrids/v2.0/docs |

> **Heads-up:** the public REST endpoint is **aggressively rate-limited** (~5 requests/min)
> and often returns 503. The helper below retries with backoff and fails gracefully.

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### 1. Resilient point query (retries on 503 / timeout)

In [2]:
import time

SOILGRIDS = "https://rest.isric.org/soilgrids/v2.0/properties/query"

def soilgrids_point(lat=LAT, lon=LON, props=("phh2o", "soc", "clay", "sand", "nitrogen", "cec"),
                    depth="0-5cm", tries=4, timeout=40):
    for i in range(tries):
        try:
            r = requests.get(SOILGRIDS, params={"lon": lon, "lat": lat, "property": list(props),
                             "depth": depth, "value": "mean"}, timeout=timeout)
            if r.status_code == 200:
                return r.json()
            print(f"  attempt {i+1}: HTTP {r.status_code} (rate-limited); backing off...")
        except requests.exceptions.RequestException as e:
            print(f"  attempt {i+1}: {type(e).__name__}; retrying...")
        time.sleep(6 * (i + 1))
    return None

sg = soilgrids_point()
print("Reached SoilGrids." if sg else "SoilGrids unavailable right now (transient) — rerun later.")

Reached SoilGrids.


### 2. Tidy the result (apply each layer's scaling `d_factor`)

In [3]:
if sg:
    rows = []
    for layer in sg["properties"]["layers"]:
        u = layer["unit_measure"]
        raw = layer["depths"][0]["values"]["mean"]
        rows.append({"property": layer["name"],
                     "value": None if raw is None else raw / u["d_factor"],
                     "units": u["target_units"], "depth": layer["depths"][0]["label"]})
    soil = pd.DataFrame(rows)
    display(soil)
else:
    print("Skipped tidy step — no data this run. The code is ready; SoilGrids just 503'd.")
    soil = pd.DataFrame()

,property,value,units,depth
0,cec,24.60,cmol(c)/kg,0-5cm
1,clay,24.80,%,0-5cm
2,nitrogen,3.08,g/kg,0-5cm
3,phh2o,6.50,-,0-5cm
4,sand,42.70,%,0-5cm
5,soc,22.90,g/kg,0-5cm


### Notes & how to extend
- **Scaling:** SoilGrids returns integers; divide by the layer's `d_factor` to get
  `target_units` (e.g. `phh2o` ÷10 = pH; `clay`/`sand` ÷10 = %; `soc` ÷10 = g/kg).
- **US work:** prefer **SSURGO** (`soil_ssurgo.ipynb`) — survey-grade and not rate-limited.
  Use SoilGrids to fill gaps or for non-US fields, and to sanity-check SSURGO.
- **Bulk / rasters:** for many points or map tiles, use the SoilGrids **WCS** service instead
  of the point REST endpoint to avoid throttling.